In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import random
import os

# --- CẤU HÌNH THƯ MỤC TẢI PDF ---
DOWNLOAD_DIR = os.path.join(os.getcwd(), "caselaw_pdfs")
if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": DOWNLOAD_DIR,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True
}
options.add_experimental_option("prefs", prefs)
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

# Khởi tạo driver (Selenium >= 4.6 sẽ tự tải driver)
driver = webdriver.Chrome(options=options)

base_url = "https://congbobanan.toaan.gov.vn/0tat1cvn/ban-an-quyet-dinh"
print(f"Đang mở trang: {base_url}")
driver.get(base_url)

print("✅ Trình duyệt đã mở. BẠN HÃY THAO TÁC TRÊN TRÌNH DUYỆT BÂY GIỜ.")
print("1. Chọn loại vụ việc: Hình sự")
print("2. Chọn khoảng thời gian")
print("3. Bấm nút TÌM KIẾM")
print("-> Khi danh sách bản án hiển thị xong, hãy chuyển sang chạy Cell 2.")

Đang mở trang: https://congbobanan.toaan.gov.vn/0tat1cvn/ban-an-quyet-dinh
✅ Trình duyệt đã mở. BẠN HÃY THAO TÁC TRÊN TRÌNH DUYỆT BÂY GIỜ.
1. Chọn loại vụ việc: Hình sự
2. Chọn khoảng thời gian
3. Bấm nút TÌM KIẾM
-> Khi danh sách bản án hiển thị xong, hãy chuyển sang chạy Cell 2.


In [7]:
# Số trang bạn muốn lấy (tính từ trang hiện tại)
num_pages = 64
caselaw_list = []

print("--- BẮT ĐẦU CRAWL LINK ---")
for page in range(1, num_pages + 1):
    print(f"Đang thu thập dữ liệu trang {page}...")
    
    try:
        # Chờ danh sách load thành công
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "//a[contains(@class, 'echo_id_pub')]"))
        )
        
        # Lấy các link
        link_elements = driver.find_elements(By.XPATH, "//a[contains(@class, 'echo_id_pub')]")
        
        for elem in link_elements:
            try:
                href = elem.get_attribute("href")
                title = elem.find_element(By.XPATH, ".//h4/span").text.strip()
                if href:
                    caselaw_list.append({"Title": title, "URL": href})
            except Exception as e:
                continue
                
        print(f"-> Đã lấy được {len(caselaw_list)} links cộng dồn.")
        
        # Chuyển trang nếu chưa đến trang cuối
        if page < num_pages:
            next_btn = driver.find_element(By.ID, "ctl00_Content_home_Public_ctl00_cmdnext")
            driver.execute_script("arguments[0].click();", next_btn)
            print("Đang chuyển trang, vui lòng đợi...")
            time.sleep(random.uniform(4, 7)) # Nghỉ để web tải trang mới
            
    except Exception as e:
        print(f"Lỗi ở trang {page}: Không tìm thấy danh sách hoặc đã hết trang.")
        break

# Lưu ra CSV
df = pd.DataFrame(caselaw_list)
df.drop_duplicates(subset=['URL'], inplace=True)
df.to_csv("caselaw_links.csv", index=False, encoding='utf-8-sig')
print(f"\n✅ Đã lưu {len(df)} link vào file caselaw_links.csv")

--- BẮT ĐẦU CRAWL LINK ---
Đang thu thập dữ liệu trang 1...
-> Đã lấy được 20 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 2...
-> Đã lấy được 40 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 3...
-> Đã lấy được 60 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 4...
-> Đã lấy được 80 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 5...
-> Đã lấy được 100 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 6...
-> Đã lấy được 120 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 7...
-> Đã lấy được 140 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 8...
-> Đã lấy được 160 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 9...
-> Đã lấy được 180 links cộng dồn.
Đang chuyển trang, vui lòng đợi...
Đang thu thập dữ liệu trang 10...
-> Đã lấy được 2

In [8]:
import os
import time
import random
import re
import urllib.parse
import unicodedata
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Set page load timeout to avoid hanging forever
driver.set_page_load_timeout(120)

# Hàm phụ trợ: Chuyển đổi tiếng Việt có dấu thành không dấu
def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return u"".join([c for c in nfkd_form if not unicodedata.combining(c)])

# Hàm phụ trợ: Tạo tên file từ Title và URL
def generate_filename(title, url):
    date_match = re.search(r"ngày\s+(\d{1,2}[/\-]\d{1,2}[/\-]\d{4})", title, re.IGNORECASE)
    date_str = date_match.group(1).replace('/', '-') if date_match else "UnknownDate"

    province_str = "UnknownProvince"
    prov_match = re.search(r"(?:tỉnh|thành phố|TP\.?|tại)\s+([A-ZĐ][\w\s]+)", title)
    if prov_match:
        raw_words = prov_match.group(1).split()
        clean_words = []
        for word in raw_words:
            if word[0].isupper() or word[0] == 'Đ':
                clean_words.append(word)
            else:
                break
        if clean_words:
            prov_name = "_".join(clean_words)
            province_str = remove_accents(prov_name).replace('Đ', 'D').replace('đ', 'd')

    parsed_url = urllib.parse.urlparse(url)
    paths = [p for p in parsed_url.path.split('/') if p]
    unique_id = paths[0] if paths else "UnknownID"

    return f"{date_str}-{province_str}-{unique_id}.pdf"

# Lấy danh sách file đã tải để skip
existing_files = set(os.listdir(DOWNLOAD_DIR))
skipped = 0

print(f"\n--- BẮT ĐẦU TẢI VÀ ĐỔI TÊN {len(df)} FILE PDF ---")
print(f"Đã có sẵn {len([f for f in existing_files if f.endswith('.pdf')])} file PDF trong thư mục.\n")

for index, row in df.iterrows():
    url = row['URL']
    title = row['Title']
    
    # Skip nếu file đã tồn tại
    expected_name = generate_filename(title, url)
    if expected_name in existing_files:
        skipped += 1
        print(f"[{index+1}/{len(df)}] Bỏ qua (đã có): {expected_name}")
        continue
    
    print(f"[{index+1}/{len(df)}] Đang xử lý: {title}")
    
    try:
        driver.get(url)
    except Exception as e:
        print(f"   -> Bỏ qua: Trang tải quá lâu (timeout). {type(e).__name__}")
        continue
    
    try:
        # Lấy danh sách file trong thư mục TRƯỚC khi tải để làm mốc so sánh
        before_files = set(os.listdir(DOWNLOAD_DIR))

        # Tìm link tải file PDF nằm trong thẻ có id="2b"
        download_elem = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//div[@id='2b']//a[contains(@href, '.pdf')]"))
        )
        pdf_url = download_elem.get_attribute("href")
        
        # Tải file bằng Selenium
        try:
            driver.get(pdf_url)
        except Exception:
            pass  # PDF download may trigger timeout — that's OK, file still downloads
        
        # ---- VÒNG LẶP CHỜ TẢI VÀ ĐỔI TÊN ----
        timeout = 30 # Chờ tối đa 30 giây cho mỗi file
        new_file = None
        
        for _ in range(timeout):
            time.sleep(1) # Chờ 1 giây rồi kiểm tra lại thư mục
            current_files = set(os.listdir(DOWNLOAD_DIR))
            new_files = current_files - before_files
            
            # Nếu có file đang tải đuôi .crdownload (Chrome chưa tải xong), tiếp tục chờ
            downloading_files = [f for f in new_files if f.endswith('.crdownload') or f.endswith('.tmp')]
            
            # Nếu có file mới xuất hiện và không còn file đang tải dở
            if new_files and not downloading_files:
                pdf_files = [f for f in new_files if f.endswith('.pdf')]
                if pdf_files:
                    new_file = pdf_files[0] # Lấy tên file gốc vừa được Chrome tải về
                    break
        
        if new_file:
            # Lấy đường dẫn file gốc
            old_file_path = os.path.join(DOWNLOAD_DIR, new_file)
            
            # Tạo tên file mới và đường dẫn mới
            new_file_name = generate_filename(title, url)
            new_file_path = os.path.join(DOWNLOAD_DIR, new_file_name)
            
            # Xóa file trùng tên (nếu có) trước khi đổi tên để tránh báo lỗi FileExistsError
            if os.path.exists(new_file_path):
                os.remove(new_file_path)
                
            # Đổi tên file
            os.rename(old_file_path, new_file_path)
            existing_files.add(new_file_name)  # Cập nhật danh sách đã tải
            print(f"   -> Đã tải và đổi tên thành: {new_file_name}")
        else:
            print("   -> Lỗi: Quá thời gian chờ 30s hoặc file tải thất bại.")

    except Exception as e:
        print("   -> Bỏ qua: Bản án này không đính kèm file PDF định dạng chuẩn.")
        
    # Nghỉ ngẫu nhiên tránh bị block IP
    time.sleep(random.uniform(2, 4)) 

print(f"\n✅ HOÀN TẤT TOÀN BỘ! Đã bỏ qua {skipped} file đã có sẵn.")
print(f"Các file PDF được lưu tại: {DOWNLOAD_DIR}")


--- BẮT ĐẦU TẢI VÀ ĐỔI TÊN 1252 FILE PDF ---
Đã có sẵn 149 file PDF trong thư mục.

[1/1252] Đang xử lý: số 268/2026/HS-ST ngày 27/11/2025 của TAND tỉnh Đồng Nai (06.03.2026)
   -> Đã tải và đổi tên thành: 27-11-2025-Dong_Nai-2ta2057184t1cvn.pdf
[2/1252] Đang xử lý: số 143/2025/HS-ST ngày 31/12/2025 của TAND tỉnh Phú Thọ (06.03.2026)
   -> Đã tải và đổi tên thành: 31-12-2025-Phu_Tho-2ta2056983t1cvn.pdf
[3/1252] Đang xử lý: số 142/2025/HS-ST ngày 30/12/2025 của TAND tỉnh Phú Thọ (06.03.2026)
   -> Đã tải và đổi tên thành: 30-12-2025-Phu_Tho-2ta2056965t1cvn.pdf
[4/1252] Đang xử lý: số 258/2025/HS-ST ngày 28/11/2025 của TAND tỉnh Đồng Nai (06.03.2026)
   -> Đã tải và đổi tên thành: 28-11-2025-Dong_Nai-2ta2056893t1cvn.pdf
[5/1252] Đang xử lý: số 255/2025/HS-ST ngày 06/11/2025 của TAND tỉnh Đồng Nai (06.03.2026)
   -> Đã tải và đổi tên thành: 06-11-2025-Dong_Nai-2ta2056899t1cvn.pdf
[6/1252] Đang xử lý: số 14/2026/HS-ST ngày 27/01/2026 của TAND tỉnh Đồng Nai (06.03.2026)
   -> Đã tải và đổi

In [4]:
# --- TIẾP TỤC TẢI TỪ LINK BỊ LỖI ---
# So sánh CSV với thư mục đã tải để tìm các file chưa có

import pandas as pd

df_all = pd.read_csv("caselaw_links.csv")

# Lấy danh sách file đã tải thành công
downloaded_files = set(os.listdir(DOWNLOAD_DIR))
downloaded_ids = set()
for f in downloaded_files:
    if f.endswith('.pdf'):
        # Trích unique_id từ tên file (phần cuối trước .pdf, sau dấu - cuối cùng)
        parts = f.replace('.pdf', '').split('-')
        if parts:
            downloaded_ids.add(parts[-1])

# Tìm các link chưa tải
pending = []
for _, row in df_all.iterrows():
    url = row['URL']
    parsed = urllib.parse.urlparse(url)
    paths = [p for p in parsed.path.split('/') if p]
    uid = paths[0] if paths else ""
    if uid not in downloaded_ids:
        pending.append(row)

df_pending = pd.DataFrame(pending)
print(f"Đã tải: {len(downloaded_ids)} | Chưa tải: {len(df_pending)} / {len(df_all)} tổng")

if len(df_pending) == 0:
    print("✅ Tất cả đã tải xong!")
else:
    # Cập nhật lại danh sách file hiện có để skip
    existing_files = set(os.listdir(DOWNLOAD_DIR))
    skipped = 0
    
    print(f"\n--- TIẾP TỤC TẢI {len(df_pending)} FILE PDF CÒN THIẾU ---")
    
    for index, row in df_pending.iterrows():
        url = row['URL']
        title = row['Title']
        
        # Skip nếu file đã tồn tại
        expected_name = generate_filename(title, url)
        if expected_name in existing_files:
            skipped += 1
            print(f"[{index+1}/{len(df_pending)}] Bỏ qua (đã có): {expected_name}")
            continue
        
        print(f"[{index+1}/{len(df_pending)}] Đang xử lý: {title}")
        
        try:
            driver.get(url)
        except Exception as e:
            print(f"   -> Bỏ qua: Trang tải quá lâu (timeout). {type(e).__name__}")
            continue
        
        try:
            before_files = set(os.listdir(DOWNLOAD_DIR))

            download_elem = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//div[@id='2b']//a[contains(@href, '.pdf')]"))
            )
            pdf_url = download_elem.get_attribute("href")
            
            try:
                driver.get(pdf_url)
            except Exception:
                pass  # PDF download may trigger timeout — file still downloads
            
            timeout_wait = 30
            new_file = None
            
            for _ in range(timeout_wait):
                time.sleep(1)
                current_files = set(os.listdir(DOWNLOAD_DIR))
                new_files = current_files - before_files
                
                downloading_files = [f for f in new_files if f.endswith('.crdownload') or f.endswith('.tmp')]
                
                if new_files and not downloading_files:
                    pdf_files = [f for f in new_files if f.endswith('.pdf')]
                    if pdf_files:
                        new_file = pdf_files[0]
                        break
            
            if new_file:
                old_file_path = os.path.join(DOWNLOAD_DIR, new_file)
                new_file_name = generate_filename(title, url)
                new_file_path = os.path.join(DOWNLOAD_DIR, new_file_name)
                
                if os.path.exists(new_file_path):
                    os.remove(new_file_path)
                    
                os.rename(old_file_path, new_file_path)
                existing_files.add(new_file_name)  # Cập nhật danh sách đã tải
                print(f"   -> Đã tải và đổi tên thành: {new_file_name}")
            else:
                print("   -> Lỗi: Quá thời gian chờ 30s hoặc file tải thất bại.")

        except Exception as e:
            print("   -> Bỏ qua: Bản án này không đính kèm file PDF định dạng chuẩn.")
            
        time.sleep(random.uniform(2, 4))

    print(f"\n✅ HOÀN TẤT! Đã bỏ qua {skipped} file đã có sẵn.")
    print(f"Kiểm tra lại thư mục: {DOWNLOAD_DIR}")

EmptyDataError: No columns to parse from file

In [9]:
driver.quit()
print("Đã đóng trình duyệt.")

Đã đóng trình duyệt.
